## Setup & Imports

In [ ]:
import sys
import yaml
import shutil
import random
from pathlib import Path

current_dir = Path.cwd()
src_dir = current_dir.parent / "scripts"
if str(src_dir) not in sys.path:
    sys.path.append(str(src_dir))

try:
    from create_patches import create_training_patches
    from repo_paths import get_repo_paths
    from data_generator import generate_synthetic_data
    from create_splits import split_dataset
except ImportError as e:
    print(f"Import Error: {e}")

## Load the Dataset Configuration

In [ ]:
# Get all the relevant paths
repo_paths = get_repo_paths()

# Get the path to the config file
config_path = repo_paths["config_dir"] / "dataset_config.yaml"

if not config_path.exists():
    raise FileNotFoundError(f"Config file not found at: {config_path}")

# Read the config file into a dictionary
with open(config_path, "r") as f:
    config = yaml.safe_load(f)


## Create the Training Patches using the Multi-Scale Sliding Window

In [ ]:
stats = create_training_patches(config=config)

print("\n--- Processing Complete ---")
print(f"Positives: {stats['pos']}")
print(f"Hard Negatives: {stats['neg_hard']}")
print(f"Backgrounds (Textured): {stats['neg_textured']}")
print(f"Backgrounds (Flat): {stats['neg_flat']}")
print(f"Discarded (Ambiguous): {stats['discarded_ambiguous']}")
print(f"Skipped (Exact Duplicates): {stats['duplicates_skipped']}")
print(f"Skipped (Perceptual Duplicates): {stats['skipped_perceptual_dup']}")

## Create the Synthetic Training Patches

In [ ]:
NUM_POSITIVES = 25_000      # Total wanted positive Patches 
                            # (if you already have 20.000, only 5.000 synthetic positive patches will be generated.)
NUM_NEGATIVES = 75_000      # Total wanted negative Patches 
                            # (if you already have 55.000, only 20.000 synthetic negative patches will be generated.)

generate_synthetic_data(
    config=config,
    num_positives=NUM_POSITIVES,
    num_negatives=NUM_NEGATIVES
)

## Split the Dataset into Train, Val and Test

In [ ]:
split_dataset(config=config)